# CX Routing Engine - Web Application Launcher

Run this notebook to launch the full FastAPI Backend and React Frontend.

In [ ]:
# 1. Install Dependencies
!pip install -qU fastapi uvicorn pydantic sqlalchemy langchain-huggingface transformers mcp

In [ ]:
import subprocess
import sys
import time
import os

# 2. Start the FastAPI Backend & Frontend UI
print('Starting FastAPI Application...')

# Force CWD to the notebook's directory so imports and frontend static files resolve correctly
notebook_dir = os.path.abspath('')

try:
    backend_process = subprocess.Popen([
        sys.executable, '-m', 'uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8001'
    ], cwd=notebook_dir, stdout=open(os.path.join(notebook_dir, 'backend_server.log'), 'w'), stderr=subprocess.STDOUT)
    time.sleep(8) # Wait slightly longer for uvicorn to boot up
    
    if backend_process.poll() is not None:
        print("\n[ERROR] Backend failed to start! Here is the error log:")
        with open(os.path.join(notebook_dir, 'backend_server.log'), 'r') as f:
            print(f.read())
    else:
        print("Backend started successfully on port 8001.")
except Exception as e:
    print(f"Failed to start backend: {e}")

In [ ]:
# 3. Expose the UI securely via Cloudflare Tunnel
import subprocess
import sys
import time
import re

print("Starting Cloudflare Tunnel...")
if sys.platform == "win32":
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe -o cloudflared.exe
    cmd = ['cloudflared.exe', 'tunnel', '--url', 'http://localhost:8001']
else:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
    cmd = ['./cloudflared', 'tunnel', '--url', 'http://localhost:8001']

cf_process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(5)
print("=" * 60)
print("🚀 CX Routing Engine Web UI is LIVE at:")
for _ in range(50):
    line = cf_process.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print(f">>> {match.group(0)} <<<")
        break
print("=" * 60)
print("Note: The first chat message will take a minute as it loads the Qwen2.5 model into your GPU.")